## Práctica de Limpieza de Texto con Pandas

### Por:
Juan Pablo Zapata Zapata

### Fecha:
2025-10-24

### Descripción:

Este notebook implementa técnicas de limpieza y normalización de texto utilizando pandas. Se trabajan diferentes tipos de datos textuales como texto general, tweets, productos y datos personales, aplicando transformaciones como normalización de mayúsculas/minúsculas, eliminación de espacios, puntuación, acentos, emojis, y extracción de elementos específicos como emails, URLs, hashtags y menciones.


## 📚 Importar librerías

In [7]:
# Librerías base para ciencia de datos
import pandas as pd
import numpy as np
import re

# Configuración de visualización
pd.set_option("display.max_colwidth", None)


## 💾 Cargar datos

In [8]:
## Conjuntos de datos de juguete
# Creamos 4 DataFrames para practicar distintas tareas de limpieza

## Texto general
df_texto = pd.DataFrame(
    {
        "texto": [
            "  ¡Hola Mundo!!!  Esto es   un EJEMPLO: visita https://miweb.com  #DataScience  :)  ",
            "Pandas > numpy? 🤔  Email: persona@example.com   ",
            "Me gusta el café colombiano; es buenísimo!!! #Café #Colombia @juan",
            "Oferta!!! 3x2 en jabón líquido 500ml - CÓDIGO: A-123",
            "      TABLAS\t, \n espacios   y saltos de línea.\r\n",
            "Teléfono: (57) 300-123-45-67; Whatsapp +57 300 222 33 44",
            "Dirección: Cll 10 # 5-20; Medellín. Barrio: La América",
            "Emoji test: 😀🙌🏽🏳️‍🌈, símbolos ©®™ y otros…",
        ]
    }
)

## Tweets
df_tweets = pd.DataFrame(
    {
        "tweet": [
            "RT @maria: Nuevo post en el blog -> http://blog.com/post?id=45 #nlp #python",
            "¡Me encanta Pandas! #datos #Python https://example.org @data_science 😊",
            "Probando cosas en Jupyter... sin link ni hashtag",
            "@juan y @ana lanzaron curso de NLP en https://cursos.ai #nlp #ml",
            "¿Pandas o Polars? debátanlo aquí 👉 https://foro.com #data",
        ]
    }
)

## Productos y precios
df_productos = pd.DataFrame(
    {
        "producto": [
            "Camisa talla M",
            "Pantalón-XL",
            "Zapato, Talla: 42",
            "Blusa s",
            "Medias 10-12",
            "Polo Talla l",
            "Vestido - 36",
            "Sombrero (talla Única)",
        ],
        "precio": [
            "$1.234,50",
            "USD 45",
            "30,00 €",
            "25.000",
            "$ 0",
            "S/. 120.90",
            "COP 9.990",
            "AR$ 2.550,00",
        ],
    }
)

## Personas, respuestas y direcciones
df_personas = pd.DataFrame(
    {
        "nombre": [
            "ana María LOPEZ",
            "Juan  perez",
            "Ñandú   Gómez",
            "Miguel (Soporte)",
            "  MÓNICA de la CRUZ  ",
            "luis-delgado",
        ],
        "categoria": ["Sí", "si", "SI ", "No", "—", None],
        "direccion": [
            "Calle 45 # 12-34, Bogotá",
            "Av. Siempre Viva 742 - Lima",
            "Cll. 10 No. 5-20 Medellín",
            "Cra 7a # 45-60, Bogotá",
            "Av. 9 #12-34  Cali",
            "Av. Insurgentes Sur 1234, CDMX",
        ],
        "id_raw": [
            "abc-0001",
            "abc 001",
            "ABC0002",
            "Abc_003",
            "abc-00004",
            "ABC-0005",
        ],
    }
)

print("df_texto:")
display(df_texto)
print("\ndf_tweets:")
display(df_tweets)
print("\ndf_productos:")
display(df_productos)
print("\ndf_personas:")
display(df_personas)


df_texto:


,texto
0,¡Hola Mundo!!! Esto es un EJEMPLO: visita https://miweb.com #DataScience :)
1,Pandas > numpy? 🤔 Email: persona@example.com
2,Me gusta el café colombiano; es buenísimo!!! #Café #Colombia @juan
3,Oferta!!! 3x2 en jabón líquido 500ml - CÓDIGO: A-123
4,"TABLAS\t, \n espacios y saltos de línea.\r\n"
5,Teléfono: (57) 300-123-45-67; Whatsapp +57 300 222 33 44
6,Dirección: Cll 10 # 5-20; Medellín. Barrio: La América
7,"Emoji test: 😀🙌🏽🏳️‍🌈, símbolos ©®™ y otros…"



df_tweets:


,tweet
0,RT @maria: Nuevo post en el blog -> http://blog.com/post?id=45 #nlp #python
1,¡Me encanta Pandas! #datos #Python https://example.org @data_science 😊
2,Probando cosas en Jupyter... sin link ni hashtag
3,@juan y @ana lanzaron curso de NLP en https://cursos.ai #nlp #ml
4,¿Pandas o Polars? debátanlo aquí 👉 https://foro.com #data



df_productos:


,producto,precio
0,Camisa talla M,"$1.234,50"
1,Pantalón-XL,USD 45
2,"Zapato, Talla: 42","30,00 €"
3,Blusa s,25.000
4,Medias 10-12,$ 0
5,Polo Talla l,S/. 120.90
6,Vestido - 36,COP 9.990
7,Sombrero (talla Única),"AR$ 2.550,00"



df_personas:


,nombre,categoria,direccion,id_raw
0,ana María LOPEZ,Sí,"Calle 45 # 12-34, Bogotá",abc-0001
1,Juan perez,si,Av. Siempre Viva 742 - Lima,abc 001
2,Ñandú Gómez,SI,Cll. 10 No. 5-20 Medellín,ABC0002
3,Miguel (Soporte),No,"Cra 7a # 45-60, Bogotá",Abc_003
4,MÓNICA de la CRUZ,—,Av. 9 #12-34 Cali,abc-00004
5,luis-delgado,None,"Av. Insurgentes Sur 1234, CDMX",ABC-0005


## 👷 Preparación de datos o Ingeniería de características

## Inspección rápida de datos

In [9]:
# Exploración de la estructura de los DataFrames
print("=== INSPECCIÓN DE DATAFRAMES ===")
print("\n1. df_texto - Info:")
df_texto.info()
print("\n2. df_tweets - Info:")
df_tweets.info()
print("\n3. df_productos - Info:")
df_productos.info()
print("\n4. df_personas - Info:")
df_personas.info()

# Observaciones:
# - df_texto: Requiere limpieza de espacios, puntuación, acentos, emojis, URLs
# - df_tweets: Necesita extracción de hashtags, menciones, URLs y limpieza de RT
# - df_productos: Requiere estandarización de tallas y normalización de precios
# - df_personas: Necesita limpieza de nombres, categorías y direcciones


=== INSPECCIÓN DE DATAFRAMES ===

1. df_texto - Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   texto   8 non-null      object
dtypes: object(1)
memory usage: 196.0+ bytes

2. df_tweets - Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   tweet   5 non-null      object
dtypes: object(1)
memory usage: 172.0+ bytes

3. df_productos - Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   producto  8 non-null      object
 1   precio    8 non-null      object
dtypes: object(2)
memory usage: 260.0+ bytes

4. df_personas - Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data

### 1) Normalización a minúsculas

In [10]:
df_texto["texto_min"] = df_texto["texto"].astype(str).str.lower()
df_texto[["texto", "texto_min"]]


,texto,texto_min
0,¡Hola Mundo!!! Esto es un EJEMPLO: visita https://miweb.com #DataScience :),¡hola mundo!!! esto es un ejemplo: visita https://miweb.com #datascience :)
1,Pandas > numpy? 🤔 Email: persona@example.com,pandas > numpy? 🤔 email: persona@example.com
2,Me gusta el café colombiano; es buenísimo!!! #Café #Colombia @juan,me gusta el café colombiano; es buenísimo!!! #café #colombia @juan
3,Oferta!!! 3x2 en jabón líquido 500ml - CÓDIGO: A-123,oferta!!! 3x2 en jabón líquido 500ml - código: a-123
4,"TABLAS\t, \n espacios y saltos de línea.\r\n","tablas\t, \n espacios y saltos de línea.\r\n"
5,Teléfono: (57) 300-123-45-67; Whatsapp +57 300 222 33 44,teléfono: (57) 300-123-45-67; whatsapp +57 300 222 33 44
6,Dirección: Cll 10 # 5-20; Medellín. Barrio: La América,dirección: cll 10 # 5-20; medellín. barrio: la américa
7,"Emoji test: 😀🙌🏽🏳️‍🌈, símbolos ©®™ y otros…","emoji test: 😀🙌🏽🏳️‍🌈, símbolos ©®™ y otros…"


### 2) Eliminación de espacios


In [11]:
df_texto["texto_espacios"] = df_texto["texto_min"].str.strip()
df_texto[["texto_espacios", "texto_min"]]


,texto_espacios,texto_min
0,¡hola mundo!!! esto es un ejemplo: visita https://miweb.com #datascience :),¡hola mundo!!! esto es un ejemplo: visita https://miweb.com #datascience :)
1,pandas > numpy? 🤔 email: persona@example.com,pandas > numpy? 🤔 email: persona@example.com
2,me gusta el café colombiano; es buenísimo!!! #café #colombia @juan,me gusta el café colombiano; es buenísimo!!! #café #colombia @juan
3,oferta!!! 3x2 en jabón líquido 500ml - código: a-123,oferta!!! 3x2 en jabón líquido 500ml - código: a-123
4,"tablas\t, \n espacios y saltos de línea.","tablas\t, \n espacios y saltos de línea.\r\n"
5,teléfono: (57) 300-123-45-67; whatsapp +57 300 222 33 44,teléfono: (57) 300-123-45-67; whatsapp +57 300 222 33 44
6,dirección: cll 10 # 5-20; medellín. barrio: la américa,dirección: cll 10 # 5-20; medellín. barrio: la américa
7,"emoji test: 😀🙌🏽🏳️‍🌈, símbolos ©®™ y otros…","emoji test: 😀🙌🏽🏳️‍🌈, símbolos ©®™ y otros…"


### 3) Eliminación de puntuación

In [12]:
df_texto["texto_sin_punct"] = df_texto["texto_espacios"].str.replace(
    r"[^\w\s]", "", regex=True
)
df_texto["texto_sin_punct"]


0    hola mundo  esto es   un ejemplo visita httpsmiwebcom  datascience  
1                                 pandas  numpy   email personaexamplecom
2             me gusta el café colombiano es buenísimo café colombia juan
3                          oferta 3x2 en jabón líquido 500ml  código a123
4                                tablas\t \n espacios   y saltos de línea
5                        teléfono 57 3001234567 whatsapp 57 300 222 33 44
6                        dirección cll 10  520 medellín barrio la américa
7                                           emoji test  símbolos  y otros
Name: texto_sin_punct, dtype: object

### 4) Eliminación de acentos

In [13]:
def quitar_acentos(text: str) -> str:
    MAP_VOCALES = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
    }
    translate = str.maketrans(MAP_VOCALES)
    text = text.translate(translate)
    return text


df_texto["texto_sin_acentos"] = df_texto["texto_sin_punct"].apply(quitar_acentos)
df_texto["texto_sin_acentos"]


0    hola mundo  esto es   un ejemplo visita httpsmiwebcom  datascience  
1                                 pandas  numpy   email personaexamplecom
2             me gusta el cafe colombiano es buenisimo cafe colombia juan
3                          oferta 3x2 en jabon liquido 500ml  codigo a123
4                                tablas\t \n espacios   y saltos de linea
5                        telefono 57 3001234567 whatsapp 57 300 222 33 44
6                        direccion cll 10  520 medellin barrio la america
7                                           emoji test  simbolos  y otros
Name: texto_sin_acentos, dtype: object

## 📊 Analysis of Results and Conclusions 

Description of the results obtained and if there are conclusions that can be drawn from them.

The analysis of results must be related to the description of the task.

**Note:** An analysis of results does not necessarily lead to conclusions, but to ideas or proposals for future work


## 💡 Proposals and Ideas

From the results obtained, what ideas or proposals can be generated to continue with the project


## 📖 References